# Run NMME Nino3.4 diagnostics

This notebook generates the NMME benchmark diagnostics used by `4_refactor_sst_skill_ts.ipynb`.

It calls `scripts/run_nmme_nino34_yeager_diag.py`, which reads the member-split NMME SST archive, computes Nino3.4 anomalies following the Yeager f03/f04-style workflow, evaluates ACC and nRMSE against HadISST2, and writes the benchmark NetCDF consumed by the skill-score overlay.

Primary output:

`/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/NMME/NMME_Nino34_skill_1982_2016.nc`

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import xarray as xr

# Identify repository root. This works when the notebook is run from either
# the repository root or the jupyter/ directory.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "scripts").exists():
    REPO_ROOT = REPO_ROOT.parent

SCRIPT_PATH = REPO_ROOT / "scripts" / "run_nmme_nino34_yeager_diag.py"
print(f"Repository root: {REPO_ROOT}")
print(f"Python         : {sys.executable}")
print(f"Script         : {SCRIPT_PATH}")

## Configuration

Use `MODEL_SET = "all"` for every downloaded SST model under the NMME archive. Use `MODEL_SET = "yeager-f03"` to reproduce the smaller eight-model Yeager-style subset. Set `FORCE = True` only when you want to rebuild cached per-model Nino3.4 anomaly files.

In [ ]:
NMME_ROOT = Path("/global/cfs/cdirs/e3sm/S2S2D/NMME/data_hindcast_by_member")
OBS_FILE = Path(
    "/global/cfs/cdirs/e3sm/diagnostics/observations/Atm/time-series/"
    "HadISST2/sst_186901_202212.nc"
)
OUTDIR = Path("/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/NMME")
FIGDIR = Path("/global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag")

OBS_VAR = "sst"
MODEL_SET = "all"  # "all" or "yeager-f03"
MODELS = []         # Optional explicit model names; leave empty to use MODEL_SET.
CLIM_START = 1982
CLIM_END = 2016
FORCE = False

EXPECTED_SKILL_FILE = OUTDIR / "NMME_Nino34_skill_1982_2016.nc"

## Validate inputs

In [ ]:
missing = []
for label, path in {
    "script": SCRIPT_PATH,
    "NMME root": NMME_ROOT,
    "HadISST2 obs file": OBS_FILE,
}.items():
    if not path.exists():
        missing.append(f"{label}: {path}")

if MODEL_SET not in {"all", "yeager-f03"}:
    missing.append(f"MODEL_SET must be 'all' or 'yeager-f03', got {MODEL_SET!r}")

if missing:
    raise FileNotFoundError("Missing or invalid configuration:\n" + "\n".join(missing))
available_models = sorted(
    p.name for p in NMME_ROOT.iterdir()
    if p.is_dir() and p.name != "logs" and (p / "sst").is_dir()
)
print(f"Downloaded NMME SST model directories: {len(available_models)}")
print(available_models)


## Run preprocessing

This can take a while on the first run because it reads each member file and writes per-model cache files under `OUTDIR/processed/`. Later runs reuse those cached files unless `FORCE = True`.

In [ ]:
cmd = [
    sys.executable,
    str(SCRIPT_PATH),
    "--nmme-root", str(NMME_ROOT),
    "--obs-file", str(OBS_FILE),
    "--obs-var", OBS_VAR,
    "--outdir", str(OUTDIR),
    "--figdir", str(FIGDIR),
    "--clim-start", str(CLIM_START),
    "--clim-end", str(CLIM_END),
]

if MODELS:
    cmd.extend(["--models", *MODELS])
else:
    cmd.extend(["--model-set", MODEL_SET])

if FORCE:
    cmd.append("--force")

env = os.environ.copy()
conda_prefix = Path(sys.prefix)
for env_name, relpath in {
    "GDAL_DATA": "share/gdal",
    "PROJ_LIB": "share/proj",
    "PROJ_DATA": "share/proj",
}.items():
    candidate = conda_prefix / relpath
    if candidate.exists():
        env[env_name] = str(candidate)

print("Running command:")
print(" ".join(cmd))
subprocess.run(cmd, cwd=REPO_ROOT, env=env, check=True)

## Inspect generated skill file

In [ ]:
if not EXPECTED_SKILL_FILE.is_file():
    raise FileNotFoundError(f"Expected skill file was not written: {EXPECTED_SKILL_FILE}")

skill_ds = xr.open_dataset(EXPECTED_SKILL_FILE)
print(EXPECTED_SKILL_FILE)
print(skill_ds)
print("models:")
print(skill_ds.model.values.tolist())